# CI/CD & Declarative Automation Bundles

Git folders, Declarative Automation Bundles (formerly Databricks Asset Bundles, DABs) and the Databricks CLI keep jobs and pipelines in Git and deploy them the same way every time. One bundle is deployed to several environments.

| Exam domain (exam guide updated August 6, 2026) | What this module covers |
|---|---|
| **Implementing CI/CD (10%)** | Git folders, `databricks.yml`, `targets`, `variables`, `mode: development` vs `production`, `presets`, `--var`, `BUNDLE_VAR_*`, `validate` / `deploy` / `run` / `destroy` |
| Working with Lakeflow Jobs (16%) | job parameters and schedules defined in YAML, paused per environment |

The CLI commands in this notebook are code examples: read them here, and run them in a terminal with the Databricks CLI when you try a bundle yourself. The verification cells run in the notebook.

> This module uses two bundles:
> - `materials/cicd_environments/`: a small job (`env_demo_job`) that writes the environment it runs in to `<catalog>.<schema>.env_demo_runs`.
> - `materials/cicd/`: the RetailHub job and pipeline that **LAB 09** deploys.

## Learning Objectives

After completing this module you will be able to:

- Explain the role of Declarative Automation Bundles (DABs) in CI/CD workflows
- Read and write a `databricks.yml` bundle configuration
- Use Git folders to connect a workspace to a Git repository
- Configure targets, variables and the deployment mode for dev, test and prod
- Deploy and run a bundle with the Databricks CLI, and override values with `--var` or `BUNDLE_VAR_*`

## CI/CD Overview for Data Engineering

CI/CD gives a data engineering team:
- the same, repeatable deployment in every environment
- code review through pull requests
- automated tests before code moves to the next environment
- a way to roll back to an earlier version

Databricks has three tools for CI/CD:

| Tool | Purpose | Storage |
|------|---------|--------|
| Git folders | Notebook version control | Git repo (GitHub, GitLab) |
| DABs (Declarative Automation Bundles) | Job/pipeline YAML definitions | `databricks.yml` |
| Databricks CLI | Deploy, run, validate bundles | Local machine / CI runner |

> **Exam Tip:** “Declarative Automation Bundles” is the current exam-guide name for the former “Databricks Asset Bundles”. The `databricks.yml` YAML format and the CLI commands are the same, and the short name DABs still applies.

## Git folders

Git folders (formerly Repos) connect a Databricks workspace folder to a Git repository.

### Setup workflow
1. **Workspace** → **Create** → **Git folder**
2. Enter the repository URL (GitHub or GitLab).
3. Choose the branch, for example `main` or `develop`.
4. Click **Pull** to get the latest changes.

### Key operations

| Action | Description |
|--------|-------------|
| `git pull` | Sync remote changes to workspace |
| Create branch | Work in isolation, then merge via PR |
| Commit & push | Save notebook changes back to Git |
| Git diff | Compare notebook versions |

<img src="../../../assets/images/nb09-git-folder-flow.png" alt="Git folders: branch, edit, commit and push, and pull in the workspace Git dialog; pull request and merge in the Git provider" width="800">

### UI walkthrough: branch, commit, push, pull request (objective 5.1)
> The course folder on the training workspace is a synced folder, not a Git folder. These steps need your own Git folder (Git credentials under **Settings → Linked accounts**).

1. In **Workspace**, open your Git folder (for example `databricks-data-engineer-associate`). Click the **branch name** button next to it to open the **Git dialog**.
2. Click **Create Branch**, enter the name `feature/<your_name>`, choose the base branch `main` and click **Create**. The folder switches to the new branch.
3. You can switch branches at any time with the branch dropdown in the same dialog. Uncommitted changes go with you to the new branch unless they conflict. If you do not want that, commit or discard them first.
4. Edit a file, for example `materials/cicd/resources/retailhub_job.yml` (change the `description`).
5. Open the Git dialog again, review the **changed files / diff** and enter a commit message.
6. Click **Commit & Push**. The commit lands on `feature/<your_name>` in the remote repo.
7. Click **Pull** in the dialog to bring in remote changes, for example after teammates merge to `main`.
8. In your Git provider (for example GitHub or GitLab), open a **pull request** from `feature/<your_name>` to `main`, review it and merge it.
9. After the merge, switch the Git folder back to `main` and click **Pull**.

### Exam Tip
> Git folders give you source control for notebooks and files in Databricks. In the workspace **Git dialog** you create and switch branches, commit & push, and pull.
> You create and merge the pull request in the **Git provider**. A CI system then runs `databricks bundle validate / deploy`.

## Declarative Automation Bundles (DABs)

DABs define Databricks resources (jobs, pipelines, clusters) as YAML configuration files. You keep these files in Git with the code, so every environment is deployed from the same definition.

### Bundle structure
```
my_bundle/
├── databricks.yml          # Root config: bundle name, workspace, targets
├── resources/
│   ├── job_medallion.yml   # Job definition
│   └── pipeline_orders.yml # Lakeflow pipeline definition
└── src/
    └── notebooks/          # Source notebooks referenced by jobs
```

### Core DABs commands

| Command | Action |
|---------|--------|
| `databricks bundle validate` | Check the configuration: YAML, variables, paths and resources |
| `databricks bundle deploy` | Deploy resources to workspace |
| `databricks bundle run <job>` | Run a deployed job |
| `databricks bundle destroy` | Remove all deployed resources |

> **Exam Tip:** `databricks bundle deploy --target prod` deploys the `prod` target configuration.

### Declarative Automation Bundles (DABs): YAML configuration

This is the YAML definition of the Medallion pipeline job in Declarative Automation Bundles (formerly Databricks Asset Bundles, DABs) format. It was exported from a working Databricks deployment.

> Before you deploy, update `notebook_path` (your Git folder location) and `parameters` to match your environment. `<git_folder>` below stands for the absolute path of your Git folder, for example `/Workspace/Users/<your_user>/databricks-data-engineer-associate`.
> The export uses absolute `/Workspace/...` paths. Inside a bundle, use relative paths instead. They are resolved relative to the YAML file that declares them, for example `../../orchestration/task_01_validate.py` in `materials/cicd/resources/retailhub_job.yml`. The notebooks are then deployed with the bundle's synced files, and the same YAML works in every target.

```yaml
resources:
  jobs:
    medallion_job:               # resource key — used by `databricks bundle run medallion_job`
      name: job_demo_medallion
      tasks:
        - task_key: bronze_customer
          notebook_task:
            notebook_path: <git_folder>/materials/medallion/bronze_customers
            source: WORKSPACE
          environment_key: Default
        - task_key: bronze_orders
          notebook_task:
            notebook_path: <git_folder>/materials/medallion/bronze_orders
            source: WORKSPACE
          environment_key: Default
        - task_key: silver_customer
          depends_on:
            - task_key: bronze_customer
          notebook_task:
            notebook_path: <git_folder>/materials/medallion/silver_customers
            source: WORKSPACE
          environment_key: Default
        - task_key: silver_orders_cleaned
          depends_on:
            - task_key: bronze_orders
          notebook_task:
            notebook_path: <git_folder>/materials/medallion/silver_orders_cleaned
            source: WORKSPACE
          environment_key: Default
        - task_key: gold_daily_orders
          depends_on:
            - task_key: silver_orders_cleaned
          notebook_task:
            notebook_path: <git_folder>/materials/medallion/gold_daily_orders
            source: WORKSPACE
          environment_key: Default
        - task_key: gold_customer_orders_summary
          depends_on:
            - task_key: silver_customer
            - task_key: gold_daily_orders
          notebook_task:
            notebook_path: <git_folder>/materials/medallion/gold_customer_orders_summary
            source: WORKSPACE
          environment_key: Default
      queue:
        enabled: true
      parameters:
        - name: catalog
          default: retailhub_<your_slug>
        - name: source_path
          default: /Volumes/retailhub_<your_slug>/default/datasets
      environments:
        - environment_key: Default
          spec:
            environment_version: "5"
      performance_target: PERFORMANCE_OPTIMIZED

```

| Element | Description |
|---------|-------------|
| `environment_key` / `environments` | Serverless jobs compute and its environment version. Alternatives: `job_cluster_key` for a job cluster, `existing_cluster_id` for an all-purpose cluster |
| `depends_on` | Defines task dependencies, which build the DAG |
| `parameters` | Job-level parameters passed to all notebooks through `dbutils.widgets` |
| `queue.enabled` | Queues new runs when the maximum number of concurrent runs is reached |
| `source: WORKSPACE` | The notebook is read from the workspace file system. Files in Git folders count as `WORKSPACE` too. Alternative: `GIT`, where the notebook is fetched from the remote repo declared in the job's `git_source` |

## Databricks CLI — Setup and Authentication

You run every bundle command with the Databricks CLI, on your own machine or on a CI runner. Install it once and log in with a profile.

> **Note:** `pip install databricks-cli` installs the legacy Python CLI (0.18.x), which has **no `bundle` command**. Bundles need the current Databricks CLI, version 1.26 or newer. This repo's bundles were last validated with v1.27.0 on 2026-11-22.

```bash
# Install the current Databricks CLI (pick one)
# macOS / Linux (Homebrew)
brew tap databricks/tap && brew install databricks
# Windows
winget install Databricks.DatabricksCLI
# Linux / macOS / WSL
curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh

databricks -v            # check the version (1.26 or newer)

# Configure authentication (OAuth, stored as a profile in ~/.databrickscfg)
databricks auth login --host https://adb-<workspace-id>.azuredatabricks.net --profile TRAINING
# (legacy alternative: databricks configure --token  -> host + personal access token)

# --- Bundle workflow ---
cd my_bundle/            # folder containing databricks.yml

# Validate configuration (variables, paths, resources)
databricks bundle validate -t dev

# Deploy to dev target
databricks bundle deploy -t dev

# Deploy to production
databricks bundle deploy -t prod

# Run a deployed job (resource key from the YAML)
databricks bundle run -t dev medallion_job
```


## The problem: same code, different environments

The job is the same in dev, test and prod, but these settings must differ:

| What changes between environments | dev | test | prod |
|---|---|---|---|
| Who deploys | the developer | CI pipeline | CI pipeline (service principal) |
| Where the files go (`root_path`) | your user folder | restricted folder | service-principal folder |
| Resource names | `[dev <you>] env_demo_job` | `[test] env_demo_job` | `env_demo_job` |
| Data location | `dev_env_demo` schema | `test_env_demo` | `prod_env_demo` |
| Schedule | paused | paused | **active** |
| Quality threshold (`min_rows`) | 1 | 100 | 1000 |

<img src="../../../assets/images/nb09-bundle-targets.png" alt="One bundle, three targets: materials/cicd_environments deployed with databricks bundle deploy -t to dev, test and prod with their names, schemas, schedule and min_rows, the variable precedence from --var to default, and the anti-pattern of three hand-edited job copies" width="800">

Copying the job three times and editing each copy by hand is an anti-pattern. A bundle keeps one YAML definition and one `targets:` block that sets the values for each environment.

## Targets and variables: multi-environment configuration

Targets hold the settings that differ per environment (dev, test, prod). `variables:` declares each value with a default, and each target sets its own value:

```yaml
bundle:
  name: medallion_pipeline

workspace:
  host: https://adb-<workspace-id>.azuredatabricks.net

variables:                     # declare variables before targets override them
  catalog:
    default: dev_catalog
  schema:
    default: bronze

targets:
  dev:
    mode: development          # prefixes resource names with '[dev <user_name>]'
    workspace:
      host: https://adb-dev.azuredatabricks.net
    variables:
      catalog: dev_catalog
      schema: bronze

  prod:
    mode: production
    workspace:
      host: https://adb-prod.azuredatabricks.net
    variables:
      catalog: prod_catalog
      schema: bronze
```

Each target can point to its own workspace (`workspace.host`). The demo bundle below keeps all targets in one workspace and changes only the values.

### Bundle anatomy: `materials/cicd_environments/`

```
materials/cicd_environments/          <- bundle root (run CLI commands here)
├── databricks.yml                    <- bundle name, variables, targets (dev / test / prod)
├── resources/env_demo_job.yml        <- the job: ${var.*} everywhere, no hard-coded env values
└── src/show_environment.py           <- task notebook: reads job parameters, writes one row per run
```

Three mechanisms work together:

| Mechanism | Where | Example from this bundle |
|---|---|---|
| Variables | `variables:` + `${var.name}` in resources | `schema: ${var.schema}`, `pause_status: ${var.schedule_pause_status}` |
| Target overrides | `targets.<t>.variables` | `test: variables: { schema: test_env_demo, min_rows: "100" }` |
| Deployment mode / presets | `targets.<t>.mode`, `presets` | `mode: development` → `[dev <user>]` prefix + paused schedules; `presets: name_prefix: "[test] "` |

Variable precedence, highest first (the first source in this list that sets a value wins): `--var="name=value"` on the CLI → `BUNDLE_VAR_name` environment variable → `.databricks/bundle/<target>/variable-overrides.json` → `targets.<t>.variables` → `default` in `variables:`.

In the `variables:` and `targets:` blocks of `databricks.yml` below, compare the values of each target with the defaults.

In [ ]:
# Print the variables and targets of the demo bundle; the job itself is in resources/env_demo_job.yml
import os

# Walk up from this notebook's folder to the repo root, so the cell also works after a move or a clone
repo = os.getcwd()
while not os.path.isfile(f"{repo}/materials/cicd_environments/databricks.yml") and repo != "/":
    repo = os.path.dirname(repo)
yml = open(f"{repo}/materials/cicd_environments/databricks.yml").read()
# Start at the variables: block and skip the YAML comments
body = yml[yml.index("\nvariables:") + 1:]
print("\n".join(l for l in body.splitlines() if not l.lstrip().startswith("#")))

## Example: validate every target and see the resolved values

`validate` shows what each target would deploy before anything reaches the workspace. Run the commands in the web terminal of this notebook. With the notebook on **Serverless**, open the bottom panel and select **Terminal** (`` Ctrl+` ``). The terminal starts in this notebook's folder, installs the Databricks CLI (tested with v1.27.0) and uses your workspace login, so the commands need no `-p` profile. Go to the bundle first with `cd ../../../materials/cicd_environments`. With a local CLI, run the same commands in `materials/cicd_environments/` of your clone and add your profile, for example `-p TRAINING`.

```bash
cd ../../../materials/cicd_environments   # web terminal: from this notebook's folder to the bundle

# Once per terminal session: your catalog, used by every command below
export CATALOG=retailhub_<your_slug>   # your catalog: CATALOG printed by 00_setup

databricks bundle validate -t dev  --var="catalog=$CATALOG"
databricks bundle validate -t test --var="catalog=$CATALOG"
databricks bundle validate -t prod --var="catalog=$CATALOG"

# See the FULLY RESOLVED configuration of one target (what would be deployed)
databricks bundle validate -t prod --var="catalog=$CATALOG" -o json
```

Result of the live run (2026-09-14):

| Target | Job name | `root_path` | Schedule | Parameters |
|---|---|---|---|---|
| dev | `[dev <you>] env_demo_job` | `/Workspace/Users/<you>/.bundle/env_demo/dev` | PAUSED | `schema=dev_env_demo`, `min_rows=1` |
| test | `[test] env_demo_job` | `/Workspace/Users/<you>/.bundle/env_demo/test` | PAUSED | `schema=test_env_demo`, `min_rows=100` |
| prod | `env_demo_job` | `/Workspace/Users/<you>/.bundle/env_demo/prod` | **UNPAUSED** | `schema=prod_env_demo`, `min_rows=1000` |

> **Note:** the job YAML never mentions dev, test or prod. `validate -o json` is the fastest way to answer *"what exactly will land in prod?"* before anything is deployed.

> **Note:** If `root_path` points to `/Workspace/Shared/...`, `validate` warns that the folder is writable by all workspace users. Production bundles go to a restricted folder, usually the service principal's home.

## Example: override values at deploy time without a YAML edit

Use an override when one deploy or one run needs a different value. CI/CD systems pass per-run values and secrets this way.

```bash
# One-off override on the command line
databricks bundle validate -t dev --var="catalog=$CATALOG" --var="min_rows=5000" \
  -o json | jq ".variables.min_rows.value"

# Same thing via environment variable — this is how CI/CD systems pass secrets / per-run values
export BUNDLE_VAR_min_rows=5000
databricks bundle validate -t dev --var="catalog=$CATALOG" -o json | jq ".variables.min_rows.value"
unset BUNDLE_VAR_min_rows
```

Both commands print `"5000"` (`jq` reads the resolved `value`; the `default` field next to it stays `"1"`). Both overrides replace the dev value of `min_rows` (`1`) for this command only, and the YAML does not change. The order of precedence is in *Bundle anatomy* above.

## Example: deploy two environments and run them

Deploy dev and test from the same definition, then run each copy once. Each run writes a row to the schema of its own environment.

```bash
databricks bundle deploy -t dev  --var="catalog=$CATALOG"
databricks bundle deploy -t test --var="catalog=$CATALOG"

databricks bundle run -t dev  --var="catalog=$CATALOG" env_demo_job
databricks bundle run -t test --var="catalog=$CATALOG" env_demo_job
```

From the web terminal, the dev deploy is source-linked. The CLI prints `Source-linked deployment is enabled` and `Files: 0 uploaded`, and the job tasks point at the notebooks in the course folder, not at copies. Bundles deployed from the workspace in development mode work this way by default (`workspace.sync_mode: in-place`), so an edit to a source notebook changes the dev job at once. The `test` target uses `mode: production`, so its deploy uploads the files to `.bundle/env_demo/test/files`. A deploy from a local CLI always uploads the files.

The job gives `CAN_VIEW` to the group in the bundle variable `training_group`. Its default is the workspace group `users`. In a delivery, pass the training group to `deploy` with `--var="training_group=<training group>"`.

Then open **Jobs & Pipelines**. You see two jobs from one definition, `[dev <user>] env_demo_job` and `[test] env_demo_job`. Each has its own parameters, tags and a paused schedule.

> **Warning: do not deploy `prod` in the training workspace.** Its schedule is `UNPAUSED` (nightly run). In a real project, CI/CD deploys prod with `run_as` set to a service principal, usually into a separate workspace (`workspace.host`).

## Verify what was deployed

The cells below use the Databricks SDK and SQL, so you do not need the CLI. Run them after the dev and test jobs are deployed and have run.

- The jobs cell lists every deployed copy. The bundle gives the training group `CAN_VIEW` on the job (`permissions` in `resources/env_demo_job.yml`), so the whole group sees the copies.
- The data cell reads the `env_demo_runs` tables in your own catalog. If you have not run the job yourself, it prints `No env_demo_runs tables in <your catalog> yet`.

In the jobs table, compare the schedule, `schema` and `min_rows` of the dev and test copies. The `dev` tag (`dev=<your user>`) comes from `mode: development`.

In the data table, each run counted up to 5,000 rows of `samples.nyctaxi.trips` (`source_rows`) and checked them against `min_rows`, a small quality gate (`gate` = PASSED or FAILED). The threshold grows from dev (1) to test (100) and prod (1000).


In [ ]:
%run ../../setup/00_setup

In [ ]:
# List every deployed copy of env_demo_job with its schedule, parameters and tags
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
rows = []
for j in w.jobs.list(expand_tasks=False):
    name = j.settings.name if j.settings else ""
    if "env_demo_job" not in name:
        continue
    s = w.jobs.get(j.job_id).settings
    params = {p.name: p.default for p in (s.parameters or [])}
    rows.append((name,
                 s.schedule.pause_status.value if s.schedule and s.schedule.pause_status else None,
                 params.get("env_name"), params.get("schema"), params.get("min_rows"),
                 ", ".join(f"{k}={v}" for k, v in sorted((s.tags or {}).items()))))

if not rows:
    print("No env_demo_job deployed yet — run "
          "`databricks bundle deploy -t dev` / `-t test`.")
else:
    schema = ("job_name string, schedule string, env_name string, schema string, "
              "min_rows string, tags string")
    display(spark.createDataFrame(rows, schema))

In [ ]:
# Each environment wrote to its own schema in your catalog (CATALOG from 00_setup)
queries = []
for schema in ["dev_env_demo", "test_env_demo", "prod_env_demo"]:
    if spark.sql(f"SHOW SCHEMAS IN {CATALOG} LIKE '{schema}'").count() and \
       spark.sql(f"SHOW TABLES IN {CATALOG}.{schema} LIKE 'env_demo_runs'").count():
        queries.append(f"SELECT '{schema}' AS schema_name, * "
                       f"FROM {CATALOG}.{schema}.env_demo_runs")

if queries:
    display(spark.sql(" UNION ALL ".join(queries) + " ORDER BY run_ts DESC"))
else:
    print(f"No env_demo_runs tables in {CATALOG} yet — "
          "run the job in at least one target first.")

## dev vs production mode: what the CLI does for you

The `mode` of a target changes several defaults at deploy time:

| Behaviour | `mode: development` | `mode: production` |
|---|---|---|
| Resource name prefix | `[dev <user_name>]` automatically | none (unless `presets.name_prefix`) |
| Schedules / triggers | **paused** by default. An explicit `pause_status` on the resource overrides it, and this bundle sets it with `${var.schedule_pause_status}` | kept as declared |
| Pipelines | `development: true` | as declared |
| Default `root_path` | `/Workspace/Users/<you>/.bundle/<bundle>/<target>` | must be set explicitly; restricted folder recommended |
| Concurrent runs / locks | relaxed | enforced |
| Who should deploy | developer | CI/CD with a **service principal** (`run_as`) |


## Example: deploy checklist for the RetailHub bundle (`materials/cicd/`)

This repo has a working bundle in `materials/cicd/`: `databricks.yml`, `resources/retailhub_pipeline.yml` and `resources/retailhub_job.yml`. Run the steps in the web terminal of this notebook (Serverless, bottom panel → **Terminal**), as in the examples above, or from a local clone of `databricks-data-engineer-associate` with the Databricks CLI (tested with v1.27.0).

1. [ ] Go to the folder that contains `databricks.yml`: `cd ../../../materials/cicd` in a new web terminal (it starts in this notebook's folder), `cd ../cicd` if the terminal is still in `materials/cicd_environments` from the examples above, `cd materials/cicd` in a local clone
2. [ ] Authenticate. The web terminal uses your workspace login, nothing to do. With a local CLI, run `databricks auth login --host <workspace-url> --profile TRAINING` and add `-p TRAINING` to the commands below.
3. [ ] `databricks bundle validate -t dev --var="catalog=retailhub_<your_slug>"` returns no errors
4. [ ] `databricks bundle deploy -t dev --var="catalog=retailhub_<your_slug>" --var="training_group=<training group>"`. The group gets `CAN_VIEW` on the job and the pipeline (default: the workspace group `users`), so the group can follow the deployment (LAB 09, Option D)
5. [ ] `databricks bundle run -t dev --var="catalog=retailhub_<your_slug>" retailhub_job`
6. [ ] Optional teardown: `databricks bundle destroy -t dev --var="catalog=retailhub_<your_slug>"`

After deployment, check:
- [ ] **Jobs & Pipelines** shows `[dev <user>] retailhub_job` and `[dev <user>] retailhub_pipeline` (prefix from `mode: development`)
- [ ] The bundle folder `/Workspace/Users/<user>/.bundle/retailhub/dev/`. A dev deploy from the web terminal is source-linked (`Files: 0 uploaded`): the job tasks point at the notebooks in `materials/orchestration` of the course folder. A deploy from a local CLI uploads copies to `.bundle/retailhub/dev/files`
- [ ] The job's pipeline task points at the bundle's pipeline (`${resources.pipelines.retailhub_pipeline.id}`)
- [ ] The job page shows the banner *Managed by Declarative Automation Bundles* and an **Enable edits** button. UI edits made after it are overwritten by the next `bundle deploy`, so change the YAML in Git instead
- [ ] LAB 09 repeats this flow step by step.


## Example: cleanup after the demo

Remove the dev and test deployments so that no demo jobs stay in the workspace.

```bash
databricks bundle destroy -t dev  --var="catalog=$CATALOG" --auto-approve
databricks bundle destroy -t test --var="catalog=$CATALOG" --auto-approve
```

`destroy` removes the jobs and the bundle files of that target. The `*_env_demo` schemas are data, not bundle resources. For a clean catalog, drop `dev_env_demo` and `test_env_demo` in `$CATALOG` with `DROP SCHEMA <catalog>.<schema> CASCADE`.

## Summary

| Topic | Key Concept | Exam Keywords |
|-------|-------------|---------------|
| Git folders | Notebook version control in workspace | Git dialog: create/switch branch, Commit & Push, Pull; PR in Git provider |
| DABs | YAML-defined jobs and pipelines | `databricks.yml`, bundle |
| Targets & variables | Per-environment values | `-t <target>`, `${var.name}`, `--var` > `BUNDLE_VAR_*` > `variable-overrides.json` > `targets.<t>.variables` > `default` |
| Deployment mode | `development` vs `production`, `presets` | `[dev <user>]` prefix, schedules paused by default |
| CLI Deploy | `bundle validate` / `deploy -t <target>` / `run` | Current Databricks CLI, tested with v1.27.0 (not `pip install databricks-cli`) |

> **Exam tips:**
> - Targets select the environment (`-t`).
> - Variables carry the per-environment values.
> - `mode: development` gives prefixed names and schedules paused by default.
> - Production deploys should run as a service principal.
> - `bundle destroy` removes everything a target deployed.


← <a href="$./08_job_orchestration">08 — Lakeflow Jobs</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$./10_governance_and_security">10 — Governance & Security →</a>

## Self-check (trainer): participants can skip this cell

In [ ]:
# Self-check (trainer): the bundle file was read and the deployed copies were listed (nothing is deployed from here)

# Bundle file: the printout starts at variables: and every target overrides the schema
assert body.startswith("variables:"), body[:40]
for target in ("dev", "test", "prod"):
    assert f"schema: {target}_env_demo" in body, f"no schema override for {target}"

# Deployed copies: every listed copy carries its parameters (no rows when nothing is deployed yet)
for job_name, _, env_name, job_schema, min_rows, _ in rows:
    assert None not in (env_name, job_schema, min_rows), job_name

print("Self-check passed")